# EGX daily LSTM - predict tomorrow

Reads the models saved by `lstm_egx.ipynb` and a fresh csv per stock, and predicts
the **next** trading day's `(close - open) %` for each one.

**What it needs**

| | |
|---|---|
| `models/EGX_<STOCK>.keras` | the trained model |
| `models/EGX_<STOCK>.json` | the recipe - window length, volume scaling, thresholds |
| `exported_files/latest/EGX_<STOCK>_1D_latest.csv` | recent daily bars |

The json is what makes this work. Features have to be rebuilt exactly as they were
at training time, so window length and volume scaling are read from it per stock
rather than assumed.

> **How many bars the csv needs.** The window itself needs `seq_len` days, but the
> trailing volume modes (`rvol`, `logz`, `rank`) also need `volume_window` days to
> warm up - 20 by default. So `--n-bars 9` is only enough for `VOLUME_NORM="month"`.
> For anything else download about 60 bars. The notebook refuses to guess and tells
> you which stocks are short.

## 1. Config

In [ ]:
MODEL_DIR = "models"                        # where the .keras and .json files are
DATA_DIR = "exported_files/latest"          # where the fresh csv files are
CSV_SUFFIX = "_latest"                      # matches pull_egx_stocks.py --suffix
EXCHANGE = "EGX"

STOCKS = None       # None = every model found in MODEL_DIR, or e.g. ["COMI", "NIPH"]

MAX_STALE_DAYS = 5  # warn if a csv's last row is older than this
SAVE_TO = "predictions.csv"

## 2. Imports

In [ ]:
import glob
import json
import os
import datetime

import numpy as np
import pandas as pd
import tensorflow as tf

print("tensorflow", tf.__version__)

## 3. Feature building

These two are copies of the training notebook's functions. They must stay identical -
if you change how features are built there, change them here too, or the model will
be fed something it was never trained on.

In [ ]:
def load_prices(path):
    """read the scraper's csv into one row per trading day"""
    df = pd.read_csv(path)

    date_col = "datetime" if "datetime" in df.columns else df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.rename(columns={date_col: "datetime"})
    df = df.sort_values("datetime").drop_duplicates("datetime").reset_index(drop=True)

    bad = df["open"] <= 0
    if bad.any():
        df = df[~bad].reset_index(drop=True)

    df["year"] = df["datetime"].dt.year
    df["month"] = df["datetime"].dt.month
    df["pct"] = (df["close"] - df["open"]) / df["open"] * 100.0
    return df


def add_volume_norm(df, mode, window=20, clip=3.0):
    """same rules as training, driven by the saved recipe rather than globals"""
    df = df.copy()
    v = df["volume"].astype("float64")
    floor = max(5, window // 2)

    if mode == "month":
        peak = df.groupby(["year", "month"])["volume"].transform("max")
        norm = v / pd.Series(peak, index=df.index).replace(0, np.nan)
        neutral = 0.0
    elif mode == "train":
        norm = v / v.max()
        neutral = 0.0
    elif mode == "rvol":
        baseline = v.rolling(window, min_periods=floor).median().shift(1)
        norm = (v / baseline.replace(0, np.nan)).clip(0.0, clip)
        neutral = 1.0
    elif mode == "logz":
        log_v = np.log1p(v)
        mu = log_v.rolling(window, min_periods=floor).mean().shift(1)
        sd = log_v.rolling(window, min_periods=floor).std().shift(1)
        norm = ((log_v - mu) / sd.replace(0, np.nan)).clip(-clip, clip)
        neutral = 0.0
    elif mode == "rank":
        norm = v.rolling(window, min_periods=floor).apply(
            lambda w: float((w[:-1] < w[-1]).mean()), raw=True)
        neutral = 0.5
    else:
        raise ValueError(f"unknown volume mode {mode!r} in the saved recipe")

    df["volume_norm"] = norm.fillna(neutral)
    return df


def classify(pct, up, down):
    pct = np.asarray(pct, dtype="float64")
    return np.where(pct >= up, "up", np.where(pct <= down, "down", "tie"))

## 4. Find what we have

A model is only usable if its `.keras`, its `.json` and a csv are all present.

In [ ]:
def available_models(model_dir=MODEL_DIR, stocks=STOCKS):
    """(stock, keras path, recipe) for every model that has its recipe beside it"""
    found = []
    for keras_path in sorted(glob.glob(os.path.join(model_dir, "*.keras"))):
        stem = os.path.splitext(os.path.basename(keras_path))[0]
        recipe_path = os.path.join(model_dir, stem + ".json")

        if not os.path.exists(recipe_path):
            print(f"{stem}: no {stem}.json beside the model, skipped - the features "
                  "cannot be rebuilt without it")
            continue

        with open(recipe_path) as f:
            recipe = json.load(f)

        stock = recipe.get("stock", stem.split("_")[-1])
        if stocks and stock not in stocks:
            continue
        found.append((stock, keras_path, recipe))

    if not found:
        raise FileNotFoundError(
            f"no usable models in {model_dir}. Upload the models.zip that "
            "lstm_egx.ipynb produced and unzip it there.")
    return found


def csv_for(stock, data_dir=DATA_DIR, suffix=CSV_SUFFIX, exchange=EXCHANGE):
    name = f"{exchange}_{stock}_1D{suffix}.csv"
    for path in [os.path.join(data_dir, name),
                 os.path.join("/content", data_dir, name),
                 os.path.join("/content/drive/MyDrive", data_dir, name),
                 name]:
        if os.path.exists(path):
            return path
    raise FileNotFoundError(
        f"{name} not found. Fetch it with:\n"
        f"  python pull_egx_stocks.py --symbols {stock} --n-bars 60 "
        f"--output-dir {data_dir} --suffix {suffix}")

## 5. Predict one stock

The window is the **last `seq_len` rows** of the csv, so the prediction is for the
first trading day after the csv's last date.

In [ ]:
def predict_one(stock, keras_path, recipe, verbose=True):
    """one next day prediction, or a row explaining why there is none"""
    seq_len = int(recipe["seq_len"])
    window = int(recipe.get("volume_window") or 20)
    clip = float(recipe.get("volume_clip") or 3.0)
    mode = recipe["volume_norm"]
    cols = recipe.get("feature_cols", ["pct", "volume_norm"])

    df = load_prices(csv_for(stock))

    # the trailing volume modes need history before the window itself
    needed = seq_len + (window if mode in ("rvol", "logz", "rank") else 0)
    if len(df) < needed:
        return {"stock": stock, "status": f"need {needed} bars, csv has {len(df)}"}

    df = add_volume_norm(df, mode=mode, window=window, clip=clip)

    last_date = df["datetime"].iloc[-1]
    stale = (pd.Timestamp.today().normalize() - last_date.normalize()).days

    # "month" scales volume by the calendar month's max. Mid month that max is
    # taken over a handful of days, so it is not the scale training used.
    if mode == "month":
        so_far = int((df["month"] == last_date.month).sum())
        if so_far < 15:
            print(f"  {stock}: volume mode 'month' but only {so_far} days of "
                  f"{last_date:%B} are in the csv - volume_norm is on a different "
                  "scale from training. Prefer rvol / logz for live prediction.")

    x = df[cols].to_numpy(dtype="float32")[-seq_len:][None, ...]
    if not np.isfinite(x).all():
        return {"stock": stock, "status": "features contain NaN or inf"}

    model = tf.keras.models.load_model(keras_path)
    if tuple(model.input_shape[1:]) != (seq_len, len(cols)):
        return {"stock": stock,
                "status": f"model wants {model.input_shape[1:]}, recipe says "
                          f"{(seq_len, len(cols))} - recipe and model disagree"}

    pred = float(model(x, training=False).numpy().ravel()[0])
    label = str(classify([pred], recipe["up_threshold"], recipe["down_threshold"])[0])

    return {
        "stock": stock,
        "status": "ok",
        "last_close_date": last_date.date().isoformat(),
        "days_old": stale,
        "predicted_pct": round(pred, 3),
        "call": label,
        "trained_rmse": recipe.get("test_rmse"),
        "beat_baseline": recipe.get("beats_baseline"),
    }

## 6. Every stock

In [ ]:
def predict_all():
    rows = []
    for stock, keras_path, recipe in available_models():
        try:
            rows.append(predict_one(stock, keras_path, recipe))
        except FileNotFoundError as e:
            rows.append({"stock": stock, "status": str(e).splitlines()[0]})

    out = pd.DataFrame(rows)
    ok = out[out["status"] == "ok"].copy()
    bad = out[out["status"] != "ok"]

    if len(ok):
        ok = ok.sort_values("predicted_pct", ascending=False)
        print(ok.to_string(index=False))
    if len(bad):
        print("\nnot predicted:")
        print(bad[["stock", "status"]].to_string(index=False))

    if len(ok):
        stale = ok[ok["days_old"] > MAX_STALE_DAYS]
        if len(stale):
            print(f"\nWARNING: stale csv for {', '.join(stale['stock'])} - "
                  f"last row more than {MAX_STALE_DAYS} days old. Re-run the "
                  "scraper before trading on this.")

        untrustworthy = ok[ok["beat_baseline"] == False]
        if len(untrustworthy):
            print(f"\nWARNING: {', '.join(untrustworthy['stock'])} did not beat the "
                  "predict-0% baseline on their test month.\n  Their predictions are "
                  "output, but there is no evidence the model has any skill.")

        ok.to_csv(SAVE_TO, index=False)
        print(f"\nsaved {SAVE_TO}")
    return out


predictions = predict_all()